In [1]:
# ci_plot.py  |  pip install numpy scipy plotly
import numpy as np
import plotly.graph_objects as go
from scipy import stats

MU, SIGMA, N, K = 100, 10, 25, 20   # true mean, sd, sample size, number of samples
T = stats.t.ppf(0.975, N - 1)

def simulate(seed):
    rng = np.random.default_rng(seed)
    x = rng.normal(MU, SIGMA, (K, N))
    means = x.mean(axis=1)
    half = T * x.std(axis=1, ddof=1) / np.sqrt(N)
    return means, half

# Pick a seed where exactly one interval misses (for a clean demo)
for seed in range(10_000):
    means, half = simulate(seed)
    miss = (means - half > MU) | (means + half < MU)
    if miss.sum() == 1:
        break

idx = np.arange(1, K + 1)
fig = go.Figure()

for mask, color, name in [(~miss, "#9aa5b1", "Contains true mean"),
                          (miss,  "#e63946", "Misses true mean")]:
    fig.add_trace(go.Scatter(
        x=idx[mask], y=means[mask], mode="markers", name=name,
        marker=dict(color="#4361ee", size=8),
        error_y=dict(type="data", array=half[mask], color=color, thickness=2, width=0),
    ))

fig.add_hline(y=MU, line=dict(color="red", dash="dash"),
              annotation_text="True mean = 100", annotation_position="top right")

fig.update_layout(
    title="95% Confidence Intervals from 20 Random Samples",
    xaxis=dict(title="Sample number", dtick=1, showgrid=False),
    yaxis=dict(title="Estimated mean", range=[88, 112]),
    template="plotly_white", height=500, legend=dict(orientation="h", y=-0.2),
)

fig.write_html("index.html", include_plotlyjs="cdn")
fig.show()